In [5]:
import requests
from bs4 import BeautifulSoup

# URL of the proceedings
url = "https://aclanthology.org/volumes/2024.wmt-1/"

# Fetch the webpage
response = requests.get(url)
soup = BeautifulSoup(response.text, 'html.parser')

# Extract all paper links
papers = soup.find_all("p", class_="d-sm-flex align-items-stretch")

for paper in papers:
    title = paper.find("strong").text.strip()
    paper_link = paper.find("a")["href"]
    
    # Visit each paper page
    paper_response = requests.get(paper_link)
    paper_soup = BeautifulSoup(paper_response.text, "html.parser")
    
    # Find authors and affiliations (varies by formatting)
    authors = paper_soup.find_all("span", class_="authors")
    for author in authors:
        name = author.text.strip()
        affiliation = author.find_next("span", class_="affiliation")
        affiliation_text = affiliation.text.strip() if affiliation else "Unknown Affiliation"
        
        print(f"Title: {title}")
        print(f"Author: {name}")
        print(f"Affiliation: {affiliation_text}")
        print("-" * 40)


In [9]:
response

<Response [200]>

In [11]:
import requests
from bs4 import BeautifulSoup

# URL of the proceedings
url = "https://aclanthology.org/volumes/2024.wmt-1/"

# Fetch the webpage
response = requests.get(url)
soup = BeautifulSoup(response.text, "html.parser")

# Find all author links (assuming structure <a href="/people/.../">Author Name</a>)
author_links = soup.find_all("a", href=lambda href: href and "/people/" in href)

# Extract and print author names
authors = [a.text.strip() for a in author_links]
author_list=[]
for author in authors:
    author_list.append(author)

In [21]:
author_list=list(set(author_list))

In [23]:
from scholarly import scholarly

# List of author names
authors = author_list
affiliations=[]
links=[]

for author in authors:
    search_query = scholarly.search_author(author)
    author_info = next(search_query, None)  # Get the first match
    
    if author_info:
        affiliations.append(author_info['affiliation'])
        links.append(f"https://scholar.google.com/citations?user={author_info['scholar_id']}")
        
    else:
        affiliations.append(None)
        links.append(None)


In [25]:
len([x for x in affiliations if x is not None]) 

437

In [30]:
import pandas as pd

df=pd.DataFrame({'authors':authors, 'affiliations':affiliations})

In [43]:
def categorize_text(text):
    if pd.isna(text):  # Handle NaN values
        return None 
    elif 'Univ' in text:
        return 'Academia'
    elif 'Doctoral' in text:
        return 'Academia'
    else:
        return None

# Apply function
df['category'] = df['affiliations'].apply(categorize_text)

In [48]:
df

,authors,affiliations,category
0,Koel Dutta Chowdhury,"Saarland University, Saarland Informatics Campus",Academia
1,Martin Bär,None,None
2,Idris Abdulmumin,"Postdoctoral Fellow, DSFSI, University of Pret...",Academia
3,Benjamin Marie,Sapienza Università di Roma,Academia
4,Lyngdoh Sarah,None,None
...,...,...,...
502,Jorge Gimenez Perez,None,None
503,Isao Goto,"Ehime University, Professor",Academia
504,Fabio Barth,DFKI,None
505,Xiaoyu He,Physikalisch-Technische Bundesanstalt (Nationa...,None


In [50]:
df.to_excel('wmt_affiliations.xlsx', index=False)